In [1]:

import os 
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

from pyspark.sql import SparkSession 
from pyspark.sql import functions as f 
from pyspark.sql.window import Window 

# local[*]: dùng tất cả CPU cores trên máy host, không cần cluster
# Phù hợp cho explore/notebook với data nhỏ như VN30
spark = SparkSession.builder \
    .appName('silver_ingestion') \
    .getOrCreate()


26/04/03 01:51:26 WARN Utils: Your hostname, thanh-Legion-5-15IAH7 resolves to a loopback address: 127.0.1.1; using 192.168.2.1 instead (on interface wlp48s0)
26/04/03 01:51:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 01:51:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
HDFS_SILVER = 'hdfs://namenode:9000/user/vn30/silver/data_clean.parquet'

data_silver = spark.read.parquet(HDFS_SILVER)
print(f'Tổng số dòng: {data_silver.count()}')
data_silver.printSchema()
data_silver.show(5)

Tổng số dòng: 47082
root
 |-- time: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)
 |-- invalid: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day: integer (nullable = true)

+----------+----+----+----+-----+-------+------+-------+-----+----+---+
|      time|open|high| low|close| volume|ticker|invalid|month|year|day|
+----------+----+----+----+-----+-------+------+-------+-----+----+---+
|2019-09-11|6.26|6.29|6.23| 6.29| 429890|   ACB|      0|    9|2019| 11|
|2019-09-12|6.32|6.41|6.29| 6.41| 622048|   ACB|      0|    9|2019| 12|
|2019-09-13|6.44|6.52|6.38| 6.52|1325067|   ACB|      0|    9|2019| 13|
|2019-09-16|6.52|6.55|6.44| 6.46|1238267|   ACB|      0|    9|2019| 16|
|2019-09-17|6.46|6.46|6.38| 6.44|1189898|   ACB|      0|    9|2019|

In [3]:
from pyspark.sql.window import Window
from pyspark.sql import functions as f

# Window theo từng ticker, sắp xếp theo ngày
w = Window.partitionBy('ticker').orderBy('time')
w_20 = Window.partitionBy('ticker').orderBy('time').rowsBetween(-20, -1)
w_7  = Window.partitionBy('ticker').orderBy('time').rowsBetween(-7, -1)

df_gold = data_silver.filter(f.col('invalid') == 0) \
    \
    .withColumn('close_1d_ago', f.lag('close', 1).over(w)) \
    .withColumn('close_7d_ago', f.lag('close', 7).over(w)) \
    \
    .withColumn('price_diff_pct_1d',
        f.round((f.col('close') - f.col('close_1d_ago')) / f.col('close_1d_ago') * 100, 2)) \
    .withColumn('price_diff_pct_1w',
        f.round((f.col('close') - f.col('close_7d_ago')) / f.col('close_7d_ago') * 100, 2)) \
    \
    .withColumn('avg_volume_20d', f.avg('volume').over(w_20)) \
    .withColumn('volume_vs_avg_20d',
        f.round(f.col('volume') / f.col('avg_volume_20d'), 2)) \
    \
    .withColumn('ma20', f.avg('close').over(w_20)) \
    .withColumn('above_ma20', f.col('close') > f.col('ma20')) \
    .withColumn('dist_from_ma20',
        f.round((f.col('close') - f.col('ma20')) / f.col('ma20') * 100, 2)) \
    \
    .drop('close_1d_ago', 'close_7d_ago', 'avg_volume_20d', 'invalid')

df_gold.printSchema()

root
 |-- time: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- price_diff_pct_1d: double (nullable = true)
 |-- price_diff_pct_1w: double (nullable = true)
 |-- volume_vs_avg_20d: double (nullable = true)
 |-- ma20: double (nullable = true)
 |-- above_ma20: boolean (nullable = true)
 |-- dist_from_ma20: double (nullable = true)



In [4]:
# Xem kết quả ngày mới nhất
latest_date = df_gold.agg(f.max('time')).collect()[0][0]
print(f'Ngày mới nhất: {latest_date}')

df_gold.filter(f.col('time') == latest_date) \
    .select('ticker', 'close', 'price_diff_pct_1d', 'price_diff_pct_1w',
            'volume_vs_avg_20d', 'above_ma20', 'dist_from_ma20') \
    .orderBy('price_diff_pct_1d', ascending=False) \
    .show(30)

Ngày mới nhất: 2026-03-27
+------+-----+-----------------+-----------------+-----------------+----------+--------------+
|ticker|close|price_diff_pct_1d|price_diff_pct_1w|volume_vs_avg_20d|above_ma20|dist_from_ma20|
+------+-----+-----------------+-----------------+-----------------+----------+--------------+
|   GVR| 32.1|              7.0|            -5.59|             0.69|     false|         -9.03|
|   PLX|42.25|             5.89|            -12.8|              0.7|     false|        -18.79|
|   MWG| 81.0|             3.18|            -3.23|             0.78|     false|         -2.18|
|   HDB| 25.3|             3.05|            -1.56|             0.96|     false|         -1.09|
|   CTG| 34.8|             2.96|            -0.57|             0.99|     false|         -0.09|
|   SSI|27.05|             2.66|            -3.39|             0.46|     false|         -7.65|
|   FPT| 76.1|             2.56|            -3.18|             0.65|     false|         -4.77|
|   BID|39.85|          